In [ ]:
#%pip install datasets
#%pip install transformers pillow
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoProcessor, AutoModel
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm # Per la barra di avanzamento
import os

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = '/content/drive/MyDrive/xai-project5/results/feature_extraction'
else:
    save_dir = os.path.abspath(os.path.join('..', 'results', 'feature_extraction'))

os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'nih_chest_embeddings.pt')
print(f"Salvataggio configurato su: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Salvataggio configurato su: /content/drive/MyDrive/xai-project5/results/feature_extraction/nih_chest_embeddings.pt


In [ ]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id="flaviagiammarino/pubmed-clip-vit-base-patch32"
processor=AutoProcessor.from_pretrained(model_id)
model=AutoModel.from_pretrained(model_id).to(device)
model.eval() #è già pre-addestrato

print("Scaricamento del dataset di radiografie in corso...")
dataset = load_dataset("g-ronimo/NIH-Chest-X-ray-dataset_10k", split="train")



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Scaricamento del dataset di radiografie in corso...


In [ ]:
dataset['labels']

Column([[0], [0], [11], [4], [11, 13]])

In [ ]:
dataset['image']

Column([<PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7D90BF0AB650>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7D90BF0A80B0>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7D90BF0ABB60>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7D90BF0ABE00>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7D90BF0AB260>])

In [ ]:
checkpoint_path = os.path.join(save_dir, 'checkpoint_embeddings.pt')

if os.path.exists(checkpoint_path):
    print("Trovato checkpoint! Riprendo l'estrazione...")
    checkpoint = torch.load(checkpoint_path)
    all_embeddings = checkpoint['embeddings']
    start_idx = checkpoint['last_index'] + 1
else:
    all_embeddings = []
    start_idx = 0

print("Estrazione embedding visivi in corso....")

with torch.no_grad():
    for idx in tqdm(range(start_idx, len(dataset))):
        item = dataset[idx]
        image=item['image']

        if image.mode!="RGB":
            image=image.convert("RGB")

        inputs=processor(images=image,return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        vision_outputs=model.get_image_features(**inputs)
        vision_tensor = vision_outputs.pooler_output
        vision_embeddings=F.normalize(vision_tensor,p=2,dim=1)
        all_embeddings.append(vision_embeddings.cpu())

        # Salviamo il checkpoint ogni 1000 immagini
        if (idx + 1) % 1000 == 0:
            torch.save({
                'embeddings': all_embeddings,
                'last_index': idx
            }, checkpoint_path)
            print(f"Checkpoint salvato al batch {idx + 1}")


dataset_tensor=torch.cat(all_embeddings,dim=0)
torch.save(dataset_tensor, save_path)

if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)

print(f"\nDataset salvato con successo. Shape: {dataset_tensor.shape}")

Estrazione embedding visivi in corso....


 13%|█▎        | 1000/7500 [05:03<40:54,  2.65it/s]

Checkpoint salvato al batch 1000


 27%|██▋       | 2000/7500 [09:22<29:08,  3.15it/s]

Checkpoint salvato al batch 2000


...

Checkpoint salvato al batch 6000


 93%|█████████▎| 7000/7500 [31:29<04:23,  1.90it/s]

Checkpoint salvato al batch 7000


100%|██████████| 7500/7500 [33:41<00:00,  3.71it/s]


Dataset salvato con successo. Shape: torch.Size([7500, 512])
